In [1]:
import pandas as pd

def read_csv_files(variables):
    """Read multiple CSV files and return a dictionary of dataframes"""
    dfs = {}
    for var in variables:
        df = pd.read_csv(f'processed_data/{var}.csv')
        # Round coordinates to ensure exact matches
        df['latitude'] = df['latitude'].round(2)
        df['longitude'] = df['longitude'].round(2)
        dfs[var] = df
    return dfs

def merge_dataframes(dfs, merge_cols):
    """Merge multiple dataframes on specified columns"""
    combined_df = None
    for df in dfs.values():
        if combined_df is None:
            combined_df = df
        else:
            # Use inner join to only keep rows where time/lat/lon match exactly
            combined_df = pd.merge(combined_df, df, 
                                 on=merge_cols,
                                 how='inner',
                                 validate='1:1')
    return combined_df.reset_index(drop=True)


In [2]:
# List of variables to process  
variables = ['thetao', 'so', 'uo', 'vo', 'wo', 'kd', 'ph', 'spco2', 'o2', 'no3', 'po4', 'si', 'fe', 'chl']

# Read all CSV files
dfs = read_csv_files(variables)

# Merge all dataframes ensuring exact matches on time/lat/lon
merge_cols = ['time', 'latitude', 'longitude'] 
combined_df = merge_dataframes(dfs, merge_cols)

# Sort by time, latitude, longitude
combined_df = combined_df.sort_values(['time', 'latitude', 'longitude'])

In [3]:
# Display first few rows and basic information
print("Combined DataFrame Shape:", combined_df.shape)
print("\nDataFrame Info:")
display(combined_df.info())

Combined DataFrame Shape: (1548, 17)

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1548 entries, 125 to 1464
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   time       1548 non-null   object 
 1   latitude   1548 non-null   float64
 2   longitude  1548 non-null   float64
 3   thetao     1548 non-null   float64
 4   so         1548 non-null   float64
 5   uo         1548 non-null   float64
 6   vo         1548 non-null   float64
 7   wo         1548 non-null   float64
 8   kd         1548 non-null   float64
 9   ph         1548 non-null   float64
 10  spco2      1548 non-null   float64
 11  o2         1548 non-null   float64
 12  no3        1548 non-null   float64
 13  po4        1548 non-null   float64
 14  si         1548 non-null   float64
 15  fe         1548 non-null   float64
 16  chl        1548 non-null   float64
dtypes: float64(16), object(1)
memory usage: 217.7+ KB


None

In [4]:
combined_df

,time,latitude,longitude,thetao,so,uo,vo,wo,kd,ph,spco2,o2,no3,po4,si,fe,chl
125,2023-12-01,20.00,87.00,27.594542,31.806778,-0.272552,-0.013161,-9.283912e-06,0.067094,8.096693,32.935770,211.99515,0.273975,0.039862,1.554029,0.000742,0.400430
74,2023-12-01,20.00,87.25,27.563759,31.831795,-0.270495,0.097522,-4.772343e-09,0.048831,8.076304,35.154930,207.69319,0.033654,0.075723,1.475166,0.000515,0.361424
97,2023-12-01,20.00,87.50,27.302694,31.519144,-0.333113,0.185945,-4.544436e-07,0.050114,8.069997,35.645756,207.43155,0.033214,0.075326,1.489294,0.000510,0.341033
12,2023-12-01,20.00,87.75,27.920725,32.762245,-0.344678,0.161340,-3.646835e-06,0.051030,8.062381,36.320953,206.83914,0.030666,0.083745,1.517367,0.000518,0.364481
52,2023-12-01,20.00,88.00,28.272133,33.357610,-0.298765,0.153842,7.304160e-07,0.055035,8.049929,37.367752,206.07112,0.039684,0.096503,1.574224,0.000530,0.403418
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1541,2024-11-01,21.75,91.00,29.354042,0.854064,0.049706,-0.029282,-1.669934e-07,0.446097,8.546706,14.635554,325.90490,96.897170,0.009941,63.728355,0.008488,7.266008
1535,2024-11-01,22.00,90.50,29.281830,1.650144,0.023765,-0.004587,5.506511e-07,0.344158,8.402761,11.133879,300.28644,55.374393,0.009514,29.423683,0.009348,5.100220
1484,2024-11-01,22.00,90.75,29.350851,0.835185,0.025352,-0.012200,-3.940682e-07,0.448890,8.517963,12.114422,340.61115,80.612250,0.010847,49.771614,0.009729,7.239828
1495,2024-11-01,22.25,91.00,29.492231,0.556050,0.010836,-0.032114,-8.930592e-07,0.426252,8.407738,37.596690,432.39725,107.424460,0.396208,35.226000,0.011815,7.854579


In [5]:
# Save the combined dataframe to CSV
combined_df.to_csv('combined_data.csv', index=False)

In [6]:
# Import numpy
import numpy as np

# Function to remove outliers from all features
def remove_outliers(data):
    clean_data = data.copy()
    
    # Get all numeric columns except time, latitude, longitude
    numeric_cols = data.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col not in ['time', 'latitude', 'longitude']]
    
    for column in numeric_cols:
        # Calculate Q1, Q3 and IQR
        Q1 = data[column].quantile(0.25)
        Q3 = data[column].quantile(0.75)
        IQR = Q3 - Q1
        
        # Create mask for non-outlier values
        mask = (data[column] >= Q1 - 1.5 * IQR) & (data[column] <= Q3 + 1.5 * IQR)
        
        # Update clean_data to keep only non-outlier values
        clean_data = clean_data[mask]
    
    # Reset index after removing outliers
    clean_data = clean_data.reset_index(drop=True)
    
    return clean_data

# Remove outliers from the combined dataframe
data = remove_outliers(combined_df)

print(f"Shape after removing outliers: {data.shape}")

Shape after removing outliers: (837, 17)


/var/folders/lw/mxt4vf712k5_lslhrjcqqj700000gn/T/ipykernel_56801/467124533.py:22: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  clean_data = clean_data[mask]
/var/folders/lw/mxt4vf712k5_lslhrjcqqj700000gn/T/ipykernel_56801/467124533.py:22: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  clean_data = clean_data[mask]
/var/folders/lw/mxt4vf712k5_lslhrjcqqj700000gn/T/ipykernel_56801/467124533.py:22: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  clean_data = clean_data[mask]
/var/folders/lw/mxt4vf712k5_lslhrjcqqj700000gn/T/ipykernel_56801/467124533.py:22: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  clean_data = clean_data[mask]
/var/folders/lw/mxt4vf712k5_lslhrjcqqj700000gn/T/ipykernel_56801/467124533.py:22: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  clean_data = clean_data[mask]
/var/folders/lw/mxt4vf712k5_lslhrjcqqj700000gn/T/i

In [7]:
data

,time,latitude,longitude,thetao,so,uo,vo,wo,kd,ph,spco2,o2,no3,po4,si,fe,chl
0,2023-12-01,20.00,89.75,27.322805,32.201420,0.049275,0.005945,-2.852281e-06,0.056219,8.079213,34.096367,210.70331,0.489192,0.022368,2.062620,0.000735,0.317517
1,2023-12-01,20.00,90.00,27.036701,31.050486,-0.196343,-0.141262,-2.424902e-07,0.055182,8.087698,33.217445,213.05751,1.284574,0.012939,2.285833,0.000848,0.283778
2,2023-12-01,20.00,90.25,27.147812,31.393084,-0.147272,-0.081226,-4.291245e-07,0.052162,8.084445,33.597878,212.06847,1.541355,0.007112,2.627164,0.000823,0.211030
3,2023-12-01,20.00,90.50,26.790709,27.032257,-0.366206,-0.024395,3.377581e-06,0.048450,8.102618,32.234962,214.02621,3.863624,0.001202,3.391350,0.001091,0.198395
4,2023-12-01,20.00,90.75,26.970106,26.796848,-0.297612,-0.190504,2.464392e-06,0.043023,8.105902,31.920260,215.08120,4.967880,0.000403,3.820824,0.001230,0.201169
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
832,2024-11-01,21.25,87.25,29.053162,15.295896,0.283282,-0.031796,1.820575e-06,0.047505,8.178094,19.154804,222.37024,5.992222,0.000404,1.838310,0.004654,0.202605
833,2024-11-01,21.25,88.50,29.194250,19.064146,0.066354,-0.137104,-8.132460e-07,0.090560,8.141164,23.431673,218.99768,4.637534,0.002718,2.381763,0.003174,0.486145
834,2024-11-01,21.25,88.75,29.763885,21.062641,0.017497,-0.016994,7.822443e-07,0.076014,8.133849,26.110245,216.79277,7.908310,0.003219,2.846911,0.003497,0.391611
835,2024-11-01,21.50,88.75,29.291569,22.154676,0.206255,-0.046011,-1.070541e-06,0.069274,8.115875,24.308140,213.93890,3.268801,0.003170,2.616735,0.004132,0.326283


In [8]:
# Save the cleaned data to a new CSV file
data.to_csv('cleaned_data.csv', index=False)

In [9]:
# Add new imports at the top of your notebook
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def create_all_timeseries(data):
    """Create time series plots for all parameters"""
    # Get numeric columns except lat/lon
    numeric_cols = data.select_dtypes(include=['float64']).columns
    parameters = [col for col in numeric_cols if col not in ['latitude', 'longitude']]
    
    # Create subplots, 3 columns and enough rows to fit all parameters
    n_cols = 2
    n_rows = (len(parameters) + n_cols - 1) // n_cols
    
    fig = make_subplots(rows=n_rows, cols=n_cols, 
                       subplot_titles=parameters)
    
    # Add each parameter's time series
    for idx, param in enumerate(parameters):
        row = idx // n_cols + 1
        col = idx % n_cols + 1
        
        fig.add_trace(
            go.Scatter(
                x=data['time'], 
                y=data[param], 
                name=param,
                mode='markers',  # Changed from default 'lines' to 'markers'
                marker=dict(
                    size=4,      # Adjust point size as needed
                    opacity=0.6  # Add some transparency to handle overlapping
                )
            ),
            row=row, col=col
        )
        
    # Update layout
    fig.update_layout(height=300*n_rows, 
                     showlegend=False,
                     title_text="Time Series of All Parameters")
    fig.show()

# Generate both visualizations
print("Time Series Plots:")
create_all_timeseries(data)

Time Series Plots:


In [10]:
# Add these imports at the top of your notebook
import folium
from ipywidgets import interact, widgets

def create_interactive_map(data):
    """Create an interactive map with parameter and time selection"""
    # Process data to get monthly averages
    data['time'] = pd.to_datetime(data['time'])
    data['month'] = data['time'].dt.to_period('M')
    monthly_data = data.groupby(['month', 'latitude', 'longitude']).mean().reset_index()
    monthly_data['month'] = monthly_data['month'].astype(str)
    
    def update_map(parameter, selected_month):
        # Create base map with scrollWheelZoom disabled
        center_lat = data['latitude'].mean()
        center_lon = data['longitude'].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=8, scrollWheelZoom=False)  # Disable scroll wheel zoom
        
        # Filter data for selected month
        month_data = monthly_data[monthly_data['month'] == selected_month]
        
        # Create colormap
        colormap = folium.LinearColormap(
            colors=['blue', 'yellow', 'red'],
            vmin=month_data[parameter].min(),
            vmax=month_data[parameter].max(),
            caption=f'{parameter} values'
        )
        
        # Add points with color based on parameter value
        for idx, row in month_data.iterrows():
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=8,
                color=None,
                fill=True,
                fill_color=colormap(row[parameter]),
                fill_opacity=0.7,
                popup=f"{parameter}: {row[parameter]:.2f}"
            ).add_to(m)
        
        # Add colormap to map
        colormap.add_to(m)
        
        # Display map
        display(m)
    
    # Get available months for dropdown
    available_months = sorted(monthly_data['month'].unique())
    
    # Create dropdown options
    numeric_cols = data.select_dtypes(include=['float64']).columns
    parameters = [col for col in numeric_cols if col not in ['latitude', 'longitude']]
    
    # Create interactive widgets
    interact(
        update_map,
        parameter=widgets.Dropdown(
            options=parameters,
            description='Parameter:',
            style={'description_width': 'initial'}
        ),
        selected_month=widgets.Dropdown(
            options=available_months,
            description='Month:',
            style={'description_width': 'initial'}
        )
    )

print("\nInteractive Map:")
create_interactive_map(data)


Interactive Map:


interactive(children=(Dropdown(description='Parameter:', options=('thetao', 'so', 'uo', 'vo', 'wo', 'kd', 'ph'…

In [11]:
# Add these imports
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time
import os

def save_map_screenshots(data):
    """Save screenshots of maps for all parameters and months"""
    # Process data to get monthly averages (same as before)
    data['time'] = pd.to_datetime(data['time'])
    data['month'] = data['time'].dt.to_period('M')
    monthly_data = data.groupby(['month', 'latitude', 'longitude']).mean().reset_index()
    monthly_data['month'] = monthly_data['month'].astype(str)
    
    # Create screenshots directory if it doesn't exist
    os.makedirs('map_screenshots', exist_ok=True)
    
    # Setup Chrome options for headless browsing
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Run in headless mode
    chrome_options.add_argument("--window-size=1920,1080")
    
    # Get parameters and months
    numeric_cols = data.select_dtypes(include=['float64']).columns
    parameters = [col for col in numeric_cols if col not in ['latitude', 'longitude']]
    available_months = sorted(monthly_data['month'].unique())
    
    # Initialize webdriver
    driver = webdriver.Chrome(options=chrome_options)
    
    try:
        for parameter in parameters:
            for month in available_months:
                # Create map
                center_lat = data['latitude'].mean()
                center_lon = data['longitude'].mean()
                m = folium.Map(location=[center_lat, center_lon], zoom_start=8)
                
                # Filter data for selected month
                month_data = monthly_data[monthly_data['month'] == month]
                
                # Create colormap
                colormap = folium.LinearColormap(
                    colors=['blue', 'yellow', 'red'],
                    vmin=month_data[parameter].min(),
                    vmax=month_data[parameter].max(),
                    caption=f'{parameter} values'
                )
                
                # Add points
                for idx, row in month_data.iterrows():
                    folium.CircleMarker(
                        location=[row['latitude'], row['longitude']],
                        radius=8,
                        color=None,
                        fill=True,
                        fill_color=colormap(row[parameter]),
                        fill_opacity=0.7,
                        popup=f"{parameter}: {row[parameter]:.2f}"
                    ).add_to(m)
                
                colormap.add_to(m)
                
                # Save map to temporary HTML file
                temp_html = f'temp_map.html'
                m.save(temp_html)
                
                # Load the HTML in selenium
                driver.get(f'file://{os.path.abspath(temp_html)}')
                time.sleep(2)  # Wait for map to load
                
                # Save screenshot
                filename = f'map_screenshots/{parameter}_{month}.png'
                driver.save_screenshot(filename)
                print(f"Saved {filename}")
                
                # Clean up temporary file
                os.remove(temp_html)
    
    finally:
        driver.quit()

# Run the function to save all screenshots
save_map_screenshots(data)

Saved map_screenshots/thetao_2023-12.png
Saved map_screenshots/thetao_2024-01.png
Saved map_screenshots/thetao_2024-02.png
Saved map_screenshots/thetao_2024-03.png
Saved map_screenshots/thetao_2024-04.png
Saved map_screenshots/thetao_2024-05.png
Saved map_screenshots/thetao_2024-06.png
Saved map_screenshots/thetao_2024-07.png
Saved map_screenshots/thetao_2024-08.png
Saved map_screenshots/thetao_2024-09.png
Saved map_screenshots/thetao_2024-10.png
Saved map_screenshots/thetao_2024-11.png
Saved map_screenshots/so_2023-12.png
Saved map_screenshots/so_2024-01.png
Saved map_screenshots/so_2024-02.png
Saved map_screenshots/so_2024-03.png
Saved map_screenshots/so_2024-04.png
Saved map_screenshots/so_2024-05.png
Saved map_screenshots/so_2024-06.png
Saved map_screenshots/so_2024-07.png
Saved map_screenshots/so_2024-08.png
Saved map_screenshots/so_2024-09.png
Saved map_screenshots/so_2024-10.png
Saved map_screenshots/so_2024-11.png
Saved map_screenshots/uo_2023-12.png
Saved map_screenshots/uo_20